In [ ]:
import logging
import os

import shared


import polars as pl
import numpy as np


import seaborn as sns
import matplotlib.pyplot as plt

from feature_engineering import build_features

In [ ]:
LOG_FMT = (
    "%(asctime)s - %(name)s [%(threadName)s] %(funcName)s [%(levelname)s] %(message)s"
)
logging.basicConfig(level=logging.INFO, format=LOG_FMT)
os.environ['RACE_TYPE'] = 've'
os.environ['FORECAST_YEAR'] = "2026"
pl.Config.set_tbl_rows(40)

In [ ]:
runs_path = f"data/long_runs_and_running_order_{shared.race_id_str()}.tsv"

runs_df = pl.read_csv(runs_path, separator="	").with_columns(
    pl.col("name").str.count_matches(" ").alias("name_space_count"),
    pl.col("name").str.split(" ").alias("_parts")
).with_columns(
    pl.col("_parts").list.slice(0, pl.col("_parts").list.len() - 1).list.join(" ").alias("first_names"),
    pl.col("_parts").list.last().alias("last_name"),
).drop("_parts")

runs_df.sort(by="name_space_count", descending=True)

In [ ]:
emit_stats = runs_df.group_by('emit').agg(
        pl.col("unique_name").n_unique().alias("num_unique_names"),
        pl.col("name").n_unique().alias("num_names"),
        pl.col("first_names").n_unique().alias("num_first_names"),
        pl.col("last_name").n_unique().alias("num_last_names"),
        pl.col("team").n_unique().alias("num_teams"),
        pl.col("emit").count().alias("num_runs"),
        pl.col("year").n_unique().alias("num_years"),
        pl.col("year").min().alias("first_year"),
        pl.col("year").max().alias("last_year"),
        pl.col("unique_name").unique().alias("unique_names"),
    ).sort(by='num_unique_names', descending=True)
emit_stats 

In [ ]:
candidates = emit_stats.filter(
    (pl.col('num_first_names') == 1) 
    & (pl.col('num_last_names') >= 2)   
    #(pl.col('num_teams') == 1) 
)
candidates

In [ ]:
from itertools import combinations
from rapidfuzz.distance import JaroWinkler

candidates.with_columns(
    pl.col("unique_names").map_elements(
        lambda names: [
            {"name_a": a, "name_b": b, "score": round(JaroWinkler.similarity(a, b), 3)}
            for a, b in combinations(names, 2)
        ],
        return_dtype=pl.List(pl.Struct({"name_a": pl.Utf8, "name_b": pl.Utf8, "score": pl.Float64}))
    ).alias("name_similarities")
).explode("name_similarities").unnest("name_similarities").sort(by='score', descending=True)